**BERN02 Exercise: Hierarchical Models and Testing**

Name: Yang Shann Wen

Date: 13 September 2026

In this exercise, we analyse a data set of a collection of 7 case-control studies examining the effectiveness of descriptive social norms on hotel customers' behavior to reuse their towels.

* **Response variable (y):** Number of customers reusing hotel towel (`reuse`)
* **Trial size (n):** Total number of customers observed (`total`)
* **Predictor (x)**: Experimental condition (`group`), where x=0 corresponds to the control group, and x=1 corresponds to the social norm group

** **
**1. Formulating regression model**

We formulate a binomial regression model, because the response variable represents counts of successes (customers who reused their towel) out of a fixed number of independent trials (total customers observed).

$$y \sim \text{Binomial}(n, p)$$
* **y**: Number of customers reusing hotel towel (`reuse`)

* **n**: Total number of customers observed (`total`)

* **p**: Probability of customer reusing their towel

****
**2. Hierarchical structure**

The regression model is a hierarchical model, as the data comes from 7 different studies conducted in different hotel settings. The baseline probability of towel reuse may vary between studies, so between-study heterogeneity should be taken into account.

A linear model with random and fixed effects is formulated:  
$$\text{logit}(p_{ij}) = \beta_0 + u_j + \beta_1 x_{ij}$$

* **$p_{ij}$:** Probability of towel reuse for group i in study j
* **$\beta_0$:** Overall baseline log-odds of towel reuse in the control group (intercept)
* **$u_j \sim\ Normal(0,τ^2)$:** Random intercept for study j, representing baseline differences across the 7 studies
* **$\beta_1$:** Fixed effect of the intervention
* **$x_{ij}$:** Indicator variable for the study group

---
**3. Parameter Estimating using Bayesian Inference**

We estimate the parameters of our hierarchical model, specifically the intervention effect $\beta_1$, using Bayesian inference via the `bambi` package.

Using the posterior distribution, we will summarize the intervention effect $\beta_1$ using the posterior mean and 90% probability interval (HDI).

In [21]:
import pandas as pd
import numpy as np
import arviz as az
import bambi as bmb
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

In [22]:
# Import data
data_file = "towelData.csv"
data = pd.read_csv(data_file, sep=';', encoding='latin1')
count = data.iloc[:, -1] # get the last column with numbers

# Data wrangling
# count has the number of yes and no for control and social norm groups
control_yes = count[::4].to_numpy() # every 4th starting from 0 - control group + yes
control_no = count[2::4].to_numpy() # every 4th starting from 2 - control group + no
control_total = np.array([y + n for y, n in zip(control_yes, control_no)])

social_yes = count[1::4].to_numpy() # every 4th starting from 1 - social norm group + yes
social_no = count[3::4].to_numpy() # every 4th starting from 3 - social norm group + no
social_total = np.array([y + n for y, n in zip(social_yes, social_no)])

study = np.arange(1,len(control_yes)+1) # 7 different studies

control_data = pd.DataFrame({"reuse": control_yes, "total": control_total, "group": "control", "study": study})
social_data = pd.DataFrame({ "reuse": social_yes, "total": social_total, "group": "social", "study": study})

combined_data = pd.concat([control_data, social_data], ignore_index=True)

# Convert data types - important for bambi that they are correctly set
# can give errors otherwise
combined_data['reuse'] = combined_data['reuse'].astype(int) # Number of reuses
combined_data['total'] = combined_data['total'].astype(int) # Total guests
combined_data['group'] = combined_data['group'].astype('category') # Group (control / social)
combined_data['study'] = combined_data['study'].astype('category') # Study number

In [23]:
# Look into the combined_data
print(combined_data)

    reuse  total    group study
0      74    211  control     1
1     103    277  control     2
2      77    135  control     3
3      82    187  control     4
4      21     25  control     5
5     123    147  control     6
6      28     30  control     7
7      98    222   social     1
8     587   1318   social     2
9     406    655   social     3
10    278    555   social     4
11     21     24   social     5
12    472    576   social     6
13    101    132   social     7


In [24]:
# For reproducibility
random_seed = 1234

In [25]:
# Formulating the hierarchical binomial model
# p(reuse, total): models the probability of reusing towels out of the total observed customers
# ~ group: fixed effect estimating the overall impact of the social norm message vs. control
# (1|study): random intercept giving each of the 7 studies its own baseline reuse rate to account for study-to-study differences

model = bmb.Model("p(reuse, total) ~ group + (1|study)", combined_data, family="binomial")
model

       Formula: p(reuse, total) ~ group + (1|study)
        Family: binomial
          Link: p = logit
  Observations: 14
        Priors: 
    target = p
        Common-level effects
            Intercept ~ Normal(mu: 0.0, sigma: 1.5)
            group ~ Normal(mu: 0.0, sigma: 1.0)
        
        Group-level effects
            1|study ~ Normal(mu: 0.0, sigma: HalfNormal(sigma: 2.5495))

In [26]:
# Fitting the model and estimating the parameter
idata_hierarchical = model.fit(random_seed=random_seed)

Output()

ERROR:pymc.stats.convergence:There were 6 divergences after tuning. Increase `target_accept` or reparameterize.


In [27]:
# Have a look at the summary of the posterior distribution with a 90% probability interval
az.summary(idata_hierarchical, hdi_prob=0.90)

Array contains NaN-value.
/usr/local/lib/python3.13/dist-packages/arviz/stats/diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
/usr/local/lib/python3.13/dist-packages/arviz/stats/diagnostics.py:991: RuntimeWarning: invalid value encountered in scalar divide
/usr/local/lib/python3.13/dist-packages/arviz/stats/diagnostics.py:595: RuntimeWarning: invalid value encountered in sqrt
/usr/local/lib/python3.13/dist-packages/arviz/stats/diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
/usr/local/lib/python3.13/dist-packages/arviz/stats/diagnostics.py:992: RuntimeWarning: invalid value encountered in sqrt
/usr/local/lib/python3.13/dist-packages/arviz/stats/diagnostics.py:595: RuntimeWarning: invalid value encountered in sqrt
/usr/local/lib/python3.13/dist-packages/arviz/stats/diagnostics.py:992: RuntimeWarning: invalid value encountered in sqrt


,mean,sd,hdi_5%,hdi_95%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
"(observed_data, p(reuse, total))",NaN,NaN,21.000,NaN,NaN,NaN,NaN,NaN,NaN
"('posterior', '1|study')[0]",-0.923,0.436,-1.624,-0.179,0.023,0.018,377.0,455.0,1.00
"('posterior', '1|study')[1]",-0.846,0.429,-1.593,-0.148,0.022,0.020,379.0,425.0,1.00
"('posterior', '1|study')[2]",-0.123,0.430,-0.882,0.531,0.022,0.019,376.0,415.0,1.01
"('posterior', '1|study')[3]",-0.614,0.431,-1.398,0.042,0.022,0.020,379.0,430.0,1.01
"('posterior', '1|study')[4]",1.156,0.529,0.253,1.983,0.023,0.017,560.0,771.0,1.00
"('posterior', '1|study')[5]",0.964,0.434,0.258,1.702,0.023,0.019,379.0,413.0,1.00
"('posterior', '1|study')[6]",0.768,0.458,0.019,1.513,0.022,0.018,425.0,562.0,1.00
"(posterior, 1|study_sigma)",1.122,0.353,0.585,1.660,0.016,0.011,482.0,607.0,1.01
"(posterior, Intercept)",0.403,0.436,-0.299,1.161,0.023,0.019,368.0,428.0,1.01


The posterior estimates of the model parameters are summarized below:

| Parameter | Posterior mean | 90% HDI |
|---|---:|---:|
| $\beta_0$ (Intercept) | 0.403 | [-0.299, 1.161] |
| $\beta_1$ (Group) | 0.210 | [0.070, 0.343] |
| $\tau$ (Study-level SD) | 1.122 | [0.585, 1.660] |

Looking at the posterior summary row for group ($\beta_1$), the estimated posterior mean is 0.210.The 90% Highest Density Interval (HDI), given by the hdi_5% and hdi_95% columns, ranges from **[0.070, 0.343]**.

Because zero is not included in this interval, the data provide strong evidence that the descriptive social norm intervention has a positive effect, increasing the odds of customers reusing their hotel towels.



**4. Formulating hypothesis**

After estimating the parameter, we need to formulate our hypothesis to test the effectiveness of the intervention.

* **Null Hypothesis ($H_0$)**: $\beta_{1}\leq 0$

  The descriptive social norm intervention has no effect, or reduces towel reuse relative to the control group.

* **Alternative Hypothesis ($H_1$)**: $\beta_{1}\ > 0$

  The descriptive social norm intervention increases towel reuse relative to the control group.

----

**5. Test the hypothesis using Bayesian Inference**

The posterior samples of the intervention effect ($\beta_{1}$), were extracted from the fitted model. The Bayesian p-value is calculated as the proportion of posterior samples for which $\beta_1\leq 0$. This represents the posterior probability that the intervention has no positive effect on towel reuse.




In [28]:
# Extract posterior samples of the intervention effect
group_samples = idata_hierarchical.posterior["group"].values.flatten()

# Calculate the Bayesian p-value
p_value = np.mean(group_samples <= 0)

print("Bayesian p-value:", p_value)

Bayesian p-value: 0.0035


As a result, the Bayesian p-value is 0.0035, meaning that only 0.35% of the posterior samples have $(\beta_1\leq0)$. This provides strong evidence against the null hypothesis and shows a positive intervention effect. Therefore, we conclude that the descriptive social norm intervention has a positive effect on towel reuse.